# Latent Reasoning Tracking — Experiments

Run all experiments for the paper. Results are saved to `figures/` and `tables/`.

**Runtime estimate on Colab T4:** ~25 minutes total.

## 1. Clone repo and install dependencies

In [ ]:
# Replace with your actual GitHub repo URL
!git clone https://github.com/MommeAbbas/latent-reasoning-tracking.git
%cd latent-reasoning-tracking

In [ ]:
# Install only what Colab doesn't already have
!pip install -q 'numpy>=1.26.0,<2.2.0' datasets scikit-learn
# Restart runtime after this cell if numpy was updated

In [ ]:
import sys
sys.path.insert(0, '.')

## 2. ROC Figure (Figure 1)

Runs the real filters across 10 seeds and plots mean ± std ROC curves.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc as sk_auc
from joblib import Parallel, delayed
from src.evaluation.run_simulation_eval import run_eval

N_SEEDS  = 10
N_TRAJ   = 150
T        = 35
BASE_FPR = np.linspace(0, 1, 101)

def process_seed(seed):
    preds_rbpf, preds_ekf, preds_pf, labels = run_eval(
        seed=seed, N_traj=N_TRAJ, T=T, PARTICLES=125, verbose=False
    )
    if len(np.unique(labels)) < 2:
        return None
    result = {}
    for name, preds in [('RBPF', preds_rbpf), ('EKF', preds_ekf), ('PF', preds_pf)]:
        fpr, tpr, _ = roc_curve(labels, preds[:, -1])
        tpr_i = np.interp(BASE_FPR, fpr, tpr); tpr_i[0] = 0.0
        result[name] = (tpr_i, sk_auc(BASE_FPR, tpr_i))
    print(f'Seed {seed} done — {labels.mean()*100:.0f}% positive')
    return result

import multiprocessing
n_jobs = multiprocessing.cpu_count()
print(f'Running {N_SEEDS} seeds across {n_jobs} CPU cores ...')

all_results = Parallel(n_jobs=n_jobs, prefer='processes')(
    delayed(process_seed)(s) for s in range(N_SEEDS)
)

tprs = {n: [] for n in ['RBPF', 'EKF', 'PF']}
aucs = {n: [] for n in ['RBPF', 'EKF', 'PF']}
for res in all_results:
    if res is None:
        continue
    for name in ['RBPF', 'EKF', 'PF']:
        tprs[name].append(res[name][0])
        aucs[name].append(res[name][1])

print('\n=== Results ===')
for name in ['RBPF', 'PF', 'EKF']:
    a = np.array(aucs[name])
    print(f'{name}: AUC = {a.mean():.3f} +/- {a.std():.3f}')

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
colors = {'RBPF': '#004488', 'PF': '#DDAA33', 'EKF': '#BB5566'}
styles = {'RBPF': '-', 'PF': '--', 'EKF': '-.'}
for name in ['RBPF', 'PF', 'EKF']:
    arr = np.array(tprs[name])
    mean_tpr, std_tpr = arr.mean(0), arr.std(0)
    mean_auc = np.array(aucs[name]).mean()
    std_auc  = np.array(aucs[name]).std()
    ax.plot(BASE_FPR, mean_tpr, color=colors[name], lw=2, ls=styles[name],
            label=f'{name} AUC={mean_auc:.2f}+/-{std_auc:.2f}')
    ax.fill_between(BASE_FPR, np.clip(mean_tpr-std_tpr,0,1),
                    np.clip(mean_tpr+std_tpr,0,1), color=colors[name], alpha=0.2)
ax.plot([0,1],[0,1],'k:',lw=1)
ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate')
ax.set_title('Early Success Prediction (SLDS Simulation)')
ax.legend(loc='lower right')
plt.tight_layout()
import os; os.makedirs('figures', exist_ok=True)
plt.savefig('figures/fig_roc_early_prediction.png', dpi=300)
plt.show()

## 3. Figure 2 — Latent Mode Detection Trace

Shows a single trajectory: raw observations, detected mode probabilities, and progress estimates per filter.

In [ ]:
# Generates figures/fig_trace_dynamics.png
!python -m src.evaluation.plot_insight_trace

## 4. Figure 3 — Particle Scaling

Shows AUC vs. particle count for RBPF vs. PF. Uses cached results from `particle_sweep_results.npy`.

In [ ]:
# Generates figures/fig_particle_scaling.png
!python -m src.evaluation.plot_particle_scaling

## 5. Ablation Study (Table 1)

In [ ]:
import src.evaluation.run_ablation_eval as abl
abl.N_SEEDS   = 5
abl.N_TRAJ    = 150
abl.PARTICLES = 125
abl.main()

## 6. Real LLM Data — collect features from Qwen2.5-Math

In [ ]:
!pip install -q transformers accelerate

In [ ]:
from src.data.gsm8k_llm_runner import run as collect_llm_data
collect_llm_data(model_name='Qwen/Qwen2.5-Math-1.5B-Instruct', n_problems=300)

## 7. Evaluate filters on real LLM traces (Table 2)

In [ ]:
from src.evaluation.run_real_eval import main as real_eval
real_eval()

## 8. Download results

In [ ]:
!zip -r results.zip figures/ tables/
from google.colab import files
files.download('results.zip')